# Crucible Python Client Tutorial

This notebook demonstrates how to use the Crucible Python client to manage datasets, samples, projects, and their relationships.

**Prerequisites:**
- Crucible API credentials configured (run `crucible config init` in terminal)
- Access to project `crucible-demo`
- Example data files in the `data/` directory

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Creating a Sample](#create-sample)
3. [Creating a Dataset](#create-dataset)
4. [Listing Datasets in a Project](#list-datasets)
5. [Getting a Dataset with Metadata](#get-dataset)
6. [Updating Dataset Metadata](#update-metadata)
7. [Downloading Dataset Files](#download-dataset)
8. [Linking a Sample to a Dataset](#link-sample-dataset)
9. [Linking Two Datasets (Parent-Child)](#link-datasets)
10. [Linking Two Samples (Parent-Child)](#link-samples)
11. [Adding a Thumbnail to a Dataset](#add-thumbnail)

<a id='setup'></a>
## 1. Setup and Configuration

First, you need to configure your Crucible API credentials. Run this command in your terminal (only needed once):

```bash
crucible config init
```

This will prompt you for:
- **API Key** - Your Crucible API key
- **API URL** - Normally leave this unset to use the client default
- **Default Project** (optional) - You can set `crucible-demo` as default

**Alternative: Without Terminal Access**

If you don't have terminal access (e.g., JupyterHub, Google Colab, VSCode Flatpak), you can initialize the client directly:

```python
client = CrucibleClient(
    api_url="https://crucible.lbl.gov/api/v3",
    api_key="your-api-key-here"
)
```

Once configured, you can import and use the Crucible client:

In [ ]:
import os
from pathlib import Path
from crucible.client import CrucibleClient

# Initialize the client (automatically loads configuration)
client = CrucibleClient()

# Get the data directory path
EXAMPLES_DIR = Path(os.getcwd())
DATA_DIR = EXAMPLES_DIR / "data"

print("Crucible client initialized successfully")
print(f"Data directory: {DATA_DIR}")

Set your project ID for this tutorial:

In [ ]:
# Project ID for this tutorial
PROJECT_ID = "crucible-demo"

# Verify project exists
project = client.projects.get(PROJECT_ID)
if project:
    print(f"Working with project: {project['project_id']}")
    print(f"  Organization: {project.get('organization', 'N/A')}")
    print(f"  Lead: {project.get('project_lead_email', 'N/A')}")
else:
    print(f"Project '{PROJECT_ID}' not found. You may need to create it first.")

<a id='create-sample'></a>
## 2. Creating a Sample

Samples represent physical materials or specimens. Let's create a sample:

In [ ]:
# Create a sample
from crucible.models import Sample

sample = client.samples.create(Sample(
    sample_name="Silicon Wafer A - Tutorial Example",
    project_id=PROJECT_ID,
    description="Silicon wafer for thermal conductivity measurements (tutorial example)",
    timestamp="2024-01-15",  # when the sample was fabricated/collected
))

sample_id = sample['unique_id']
print("Sample created successfully")
print(f"  Sample ID: {sample_id}")
print(f"  Sample Name: {sample['sample_name']}")
print(f"  Project: {sample['project_id']}")
print(f"  Timestamp: {sample.get('timestamp', 'N/A')}")

<a id='create-dataset'></a>
## 3. Creating a Dataset

Dataset records and associated files are separate API resources. This tutorial creates the dataset record first and then adds files using its MFID. Passing `files` to `client.datasets.create()` remains available as a convenience that performs the same operations in sequence.

### 3.1 Create Dataset with Metadata Only

In [ ]:
from crucible.models import Dataset

# Define dataset metadata
dataset_metadata = Dataset(
    project_id=PROJECT_ID,
    measurement="thermal_conductivity",
    dataset_name="Thermal Conductivity Measurement - Sample A (Tutorial)",
    timestamp="2024-01-15T10:30:00",  # when the measurement was taken
    public=False
)

# Create dataset without files
result = client.datasets.create(
    dataset=dataset_metadata,
    scientific_metadata={
        "temperature_range": "273-363 K",
        "measurement_method": "3-omega method",
        "equipment": "Lakeshore 336 + SR830 lock-in",
        "sample_type": "silicon wafer"
    },
    keywords=["thermal", "conductivity", "silicon", "tutorial"]
)

dataset_mfid = result['dataset_mfid']
dataset_id = dataset_mfid
print("Dataset created successfully")
print(f"  Dataset ID: {dataset_id}")
print(f"  Dataset Name: {result['created_record']['dataset_name']}")
print(f"  Timestamp: {result['created_record'].get('timestamp', 'N/A')}")

### 3.2 Manage Dataset Files

The following example discovers a directory tree, checks for duplicate basenames, and uploads the files to the existing dataset. Crucible does not preserve the local directory hierarchy, so two source files with the same basename must be resolved before uploading. One file is deliberately held back and added afterward to demonstrate adding new data later.

Replacing a file is explicit: identify and delete the old associated-file record, then add the replacement. File deletion is irreversible, so verify the dataset, file MFID, and filename first. Re-uploading means adding files to the existing dataset MFID, not creating another dataset. Unchanged files are deduplicated by SHA-256.

In [ ]:
from collections import Counter

source_files = sorted(path for path in DATA_DIR.rglob("*") if path.is_file())
duplicate_names = [
    name
    for name, count in Counter(path.name for path in source_files).items()
    if count > 1
]
if duplicate_names:
    raise ValueError(f"Duplicate filenames in source tree: {duplicate_names}")

additional_file = DATA_DIR / "measurement_notes.txt"
initial_files = [path for path in source_files if path != additional_file]

for path in initial_files:
    client.datasets.add_file(dataset_mfid, str(path))

additional_result = client.datasets.add_file(dataset_mfid, str(additional_file))
print(f"Added later: {additional_result['associated_file']['mfid']}")

current_files = client.datasets.list_files(dataset_mfid)
old_file = next(file for file in current_files if file['filename'] == additional_file.name)
client.files.delete(old_file['mfid'])
replacement_result = client.datasets.add_file(dataset_mfid, str(additional_file))
print(f"Replacement file: {replacement_result['associated_file']['mfid']}")

for path in source_files:
    client.datasets.add_file(dataset_mfid, str(path))

dataset_with_files_id = dataset_mfid
print(f"Re-uploaded files to existing dataset: {dataset_with_files_id}")

<a id='list-datasets'></a>
## 4. Listing Datasets in a Project

Retrieve all datasets associated with a project:

In [ ]:
# List all datasets in the project
datasets = client.datasets.list(project_id=PROJECT_ID, limit=50)

print(f"Found {len(datasets)} dataset(s) in project {PROJECT_ID}\n")

# Display first 5 datasets
for i, ds in enumerate(datasets[:5], 1):
    print(f"{i}. {ds.get('unique_id', 'N/A')}")
    print(f"   Name: {ds.get('dataset_name', 'Unnamed')}")
    print(f"   Measurement: {ds.get('measurement', 'N/A')}")
    print(f"   Public: {ds.get('public', False)}")
    if ds.get('creation_time'):
        print(f"   Created: {ds['creation_time'][:10]}")
    print()

<a id='get-dataset'></a>
## 5. Getting a Dataset with Metadata

Retrieve detailed information about a specific dataset, including scientific metadata:

In [ ]:
# Get dataset with metadata
dataset_details = client.datasets.get(
    dataset_mfid=dataset_mfid,
    include_metadata=True
)

print(f"Dataset: {dataset_details['unique_id']}")
print(f"Name: {dataset_details.get('dataset_name', 'N/A')}")
print(f"Measurement: {dataset_details.get('measurement', 'N/A')}")
print(f"Public: {dataset_details.get('public', False)}")
print(f"Project: {dataset_details.get('project_id', 'N/A')}")

print(f"\nScientific Metadata:")
if 'scientific_metadata' in dataset_details and dataset_details['scientific_metadata']:
    for key, value in dataset_details['scientific_metadata'].items():
        print(f"  {key}: {value}")
else:
    print("  No metadata available")

You can also get keywords and other dataset properties:

In [ ]:
# Get keywords
keywords = client.datasets.get_keywords(dataset_mfid)
if keywords:
    keyword_list = [kw.get('keyword', '') for kw in keywords]
    print(f"Keywords: {', '.join(keyword_list)}")
else:
    print("Keywords: None")

# Get thumbnails
thumbnails = client.datasets.get_thumbnails(dataset_mfid)
print(f"Number of thumbnails: {len(thumbnails)}")

<a id='update-metadata'></a>
## 6. Updating Dataset Metadata

You can update scientific metadata for an existing dataset:

In [ ]:
# Update scientific metadata for the dataset
updated_metadata = {
    "temperature_range": "273-363 K",
    "measurement_method": "3-omega method",
    "equipment": "Lakeshore 336 + SR830 lock-in",
    "sample_type": "silicon wafer",
    "calibration_date": "2026-02-20",
    "operator": "Tutorial User"
}

result = client.datasets.update_scientific_metadata(
    resource_mfid=dataset_mfid,
    metadata=updated_metadata
)

print(f"Scientific metadata updated for dataset {dataset_id}")

# Verify the update
dataset_updated = client.datasets.get(dataset_mfid, include_metadata=True)
print(f"\nUpdated Scientific Metadata:")
if dataset_updated.get('scientific_metadata'):
    sci_meta = dataset_updated['scientific_metadata']
    for key, value in sci_meta.items():
        print(f"  {key}: {value}")

<a id='download-dataset'></a>
## 7. Downloading Dataset Files

You can download files from datasets that have been ingested:

In [ ]:
# Get download links for dataset files
download_links = client.datasets.get_download_links(dataset_with_files_id)

print(f"Download links for dataset {dataset_with_files_id}:\n")
for file_mfid, url in download_links.items():
    print(f"  File MFID: {file_mfid}")
    print(f"  URL: {url}...")
    print()

# Download all files from the dataset to a directory
import tempfile
download_dir = Path(tempfile.mkdtemp())

print(f"Downloading files to: {download_dir}")
downloaded_files = client.datasets.download(
    dataset_mfid=dataset_with_files_id,
    output_dir=str(download_dir)
)

print(f"\nDownloaded {len(downloaded_files)} file(s):")
for file_path in downloaded_files:
    print(f"  - {Path(file_path).name}")

<a id='link-sample-dataset'></a>
## 8. Linking a Sample to a Dataset

Associate a dataset with a sample to indicate which sample the data comes from:

In [ ]:
# Link sample to dataset
result = client.samples.link_dataset(
    sample_mfid=sample_id,
    dataset_mfid=dataset_mfid
)

print(f"Sample {sample_id} linked to dataset {dataset_id}")

# Verify the link
datasets_for_sample = client.datasets.list(sample_mfid=sample_id)
print(f"\nDatasets linked to sample {sample_id}: {len(datasets_for_sample)}")
for ds in datasets_for_sample:
    print(f"  - {ds['unique_id']}: {ds.get('dataset_name', 'N/A')}")

<a id='link-datasets'></a>
## 9. Linking Two Datasets (Parent-Child)

Create hierarchical relationships between datasets. For example, link a processed dataset to its raw data:

In [ ]:
# Create a second dataset (processed data)
processed_dataset = Dataset(
    project_id=PROJECT_ID,
    measurement="thermal_conductivity_analysis",
    dataset_name="Processed Thermal Conductivity Data (Tutorial)",
    public=False
)

result_processed = client.datasets.create(
    dataset=processed_dataset,
    scientific_metadata={
        "thermal_conductivity_300K": 148.5,
        "thermal_conductivity_unit": "W/m·K",
        "processing_method": "polynomial curve fitting (order 2)",
        "uncertainty": 1.8,
        "parent_dataset": dataset_with_files_id
    },
    keywords=["processed", "thermal", "conductivity", "analysis", "tutorial"]
)

processed_dataset_id = result_processed['dataset_mfid']
print(f"Processed dataset created: {processed_dataset_id}")

# Link datasets: raw data (parent) -> processed data (child)
link_result = client.datasets.link(
    parent_mfid=dataset_with_files_id,
    child_mfid=processed_dataset_id
)

print("\nDatasets linked successfully")
print(f"  Parent (raw data): {dataset_with_files_id}")
print(f"  Child (processed): {processed_dataset_id}")

# List children of parent dataset
children = client.datasets.list_children(dataset_with_files_id)
print(f"\nChild datasets of {dataset_with_files_id}: {len(children)}")
for child in children:
    print(f"  - {child['unique_id']}: {child.get('dataset_name', 'N/A')}")

# List parents of child dataset
parents = client.datasets.list_parents(processed_dataset_id)
print(f"\nParent datasets of {processed_dataset_id}: {len(parents)}")
for parent in parents:
    print(f"  - {parent['unique_id']}: {parent.get('dataset_name', 'N/A')}")

In [ ]:
# Create a subsample
subsample = client.samples.create(Sample(
    sample_name="Silicon Wafer A - Region 1 (Tutorial)",
    project_id=PROJECT_ID,
    description="Sub-region of wafer A for localized measurements (tutorial example)"
))

subsample_id = subsample['unique_id']
print(f"Subsample created: {subsample_id}")

# Link samples: parent sample -> subsample
link_result = client.samples.link(
    parent_mfid=sample_id,
    child_mfid=subsample_id
)

print("\nSamples linked successfully")
print(f"  Parent: {sample_id}")
print(f"  Child: {subsample_id}")

# List children of parent sample
children = client.samples.list_children(sample_id)
print(f"\nChild samples of {sample_id}: {len(children)}")
for child in children:
    print(f"  - {child['unique_id']}: {child.get('sample_name', 'N/A')}")

# List parents of child sample
parents = client.samples.list_parents(subsample_id)
print(f"\nParent samples of {subsample_id}: {len(parents)}")
for parent in parents:
    print(f"  - {parent['unique_id']}: {parent.get('sample_name', 'N/A')}")

<a id='link-samples'></a>
## 10. Linking Two Samples (Parent-Child)

Create hierarchical relationships between samples. For example, link a subsample to its parent sample:

In [ ]:
# Path to thumbnail image
thumbnail_path = str(DATA_DIR / "thermal_measurement_preview.png")

# Verify file exists
if Path(thumbnail_path).exists():
    print(f"Thumbnail file found: {Path(thumbnail_path).name}")
    
    # Add thumbnail to dataset
    result = client.datasets.add_thumbnail(
        dataset_mfid=dataset_mfid,
        image=thumbnail_path,
        thumbnail_name="thermal_measurement_preview"
    )
    
    print(f"\nThumbnail added to dataset {dataset_id}")
    
    # List all thumbnails for the dataset
    thumbnails = client.datasets.get_thumbnails(dataset_mfid)
    print(f"\nThumbnails for dataset:")
    for thumb in thumbnails:
        print(f"  - {thumb.get('thumbnail_name', 'unnamed')}")
else:
    print(f"Thumbnail file not found: {thumbnail_path}")

<a id='add-thumbnail'></a>
## 11. Adding a Thumbnail to a Dataset

Upload a thumbnail image to provide a visual preview of your dataset:

In [ ]:
# Display all IDs created
print("Resource IDs created in this tutorial:")
print(f"\nSamples:")
print(f"  sample_id = {sample_id}")
print(f"  subsample_id = {subsample_id}")
print(f"\nDatasets:")
print(f"  dataset_id = {dataset_id}")
print(f"  dataset_with_files_id = {dataset_with_files_id}")
print(f"  processed_dataset_id = {processed_dataset_id}")

print("\n" + "="*60)
print("You can open any of these resources in your browser using:")
print("="*60)
print(f"\n  crucible open {sample_id}")
print(f"  crucible open {dataset_id}")
print(f"  crucible open {dataset_with_files_id}")
print("\nThe 'crucible open' command works with any dataset, sample, or project ID.")

## Summary

This notebook demonstrated the core Crucible operations:

1. **Configuration** - Set up API credentials with `crucible config init`
2. **Create Sample** - `client.samples.create()`
3. **Create Dataset and Manage Files** - `client.datasets.create()`, `client.datasets.add_file()`, and `client.files.delete()`
4. **List Datasets** - `client.datasets.list(project_id=...)`
5. **Get Dataset Details** - `client.datasets.get(dataset_mfid, include_metadata=True)`
6. **Update Dataset Metadata** - `client.datasets.update_scientific_metadata()`
7. **Download Dataset Files** - `client.datasets.get_download_links()` and `client.datasets.download()`
8. **Link Sample to Dataset** - `client.samples.link_dataset(sample_mfid, dataset_mfid)`
9. **Link Datasets** - `client.datasets.link()`, `list_children()`, `list_parents()`
10. **Link Samples** - `client.samples.link()`, `list_children()`, `list_parents()`
11. **Add Thumbnail** - `client.datasets.add_thumbnail()`

### Resource IDs Created in This Tutorial

The following variables contain IDs of resources created in this notebook:

In [ ]:
# List all samples in a project
samples = client.samples.list(project_id=PROJECT_ID, limit=100)
print(f"Total samples returned for project: {len(samples)}")

# List accessible projects
projects = client.projects.list(limit=100)
print(f"Total projects returned: {len(projects)}")

### Additional Resources

- **Documentation**: See `crucible/cli/README.md` for CLI usage
- **API Reference**: Check docstrings in `crucible/resources/` for all available methods
- **Parsers**: See `crucible/parsers/README.md` for how to use and extend parsers (BaseParser, LAMMPSParser, MatEnsembleManagerParser, MatEnsembleRunParser)
- **Data Files**: Example data files used in this tutorial are in `examples/data/`